# Exploring the Hackathon Data

This notebook walks through every data source available for the hackathon, shows the schema, and demonstrates how to load, visualize, and merge the datasets.

**Prerequisites:** Run `python download_data.py` first to populate `data/train/`.

## Section 1: Setup & Data Loading

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import backtester modules if needed
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import glob
import sqlite3

import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# Change this to "data/validation/" when testing on the validation set
DATA_DIR = Path("data/train/")

# Path to the SQLite database (contains market_prices, rtds_prices, market_outcomes)
DB_PATH = DATA_DIR / "polymarket.db"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR.resolve()}")
print(f"DB exists:    {DB_PATH.exists()}")

In [ ]:
# Connect to the database and list all tables
conn = sqlite3.connect(str(DB_PATH))
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tables in polymarket.db:")
print(tables.to_string(index=False))

## Section 2: Market Prices (Polymarket)

The `market_prices` table contains YES/NO prices, bids, asks, volume, and liquidity for every active BTC market, sampled roughly once per second.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in **microseconds** (divide by 1e6 for seconds) |
| `interval` | Market interval: `"5m"`, `"15m"`, or `"hourly"` |
| `market_slug` | Unique market identifier (e.g. `btc-updown-15m-1717243200`) |
| `yes_price` | Last traded YES token price (0-1) |
| `no_price` | Last traded NO token price (0-1) |
| `yes_bid` | Best bid for YES token |
| `yes_ask` | Best ask for YES token |
| `no_bid` | Best bid for NO token |
| `no_ask` | Best ask for NO token |
| `volume` | Cumulative market volume (USD) |
| `liquidity` | Current market liquidity (USD) |

In [ ]:
# Preview the market_prices table
prices_sample = pd.read_sql_query("SELECT * FROM market_prices LIMIT 5", conn)
prices_sample

In [ ]:
# Count rows per interval
interval_counts = pd.read_sql_query(
    "SELECT interval, COUNT(*) AS row_count FROM market_prices GROUP BY interval",
    conn,
)
interval_counts

In [ ]:
# Plot YES price over time for one 15-minute market
# Pick the first 15m market slug that has enough data
slug_row = pd.read_sql_query(
    """SELECT market_slug, COUNT(*) AS cnt
       FROM market_prices
       WHERE interval = '15m'
       GROUP BY market_slug
       ORDER BY cnt DESC
       LIMIT 1""",
    conn,
)
example_slug = slug_row["market_slug"].iloc[0]
print(f"Plotting market: {example_slug}")

market_df = pd.read_sql_query(
    "SELECT timestamp_us, yes_price FROM market_prices WHERE market_slug = ? ORDER BY timestamp_us",
    conn,
    params=[example_slug],
)

# Convert microseconds to datetime
market_df["time"] = pd.to_datetime(market_df["timestamp_us"] / 1e6, unit="s", utc=True)

plt.figure(figsize=(12, 4))
plt.plot(market_df["time"], market_df["yes_price"], linewidth=0.8)
plt.title(f"YES Price - {example_slug}")
plt.xlabel("Time (UTC)")
plt.ylabel("YES Price")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 3: Chainlink Oracle Prices (rtds_prices)

The `rtds_prices` table stores on-chain Chainlink BTC/USD oracle prices. **This is the reference price used for market settlement** -- if Chainlink BTC at close >= Chainlink BTC at open, YES wins.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in **microseconds** |
| `source` | Price source (filter for `"chainlink"`) |
| `symbol` | Asset symbol (e.g. `"BTC"`) |
| `price` | BTC/USD price from Chainlink oracle |

In [ ]:
# Preview Chainlink prices
chainlink_sample = pd.read_sql_query(
    "SELECT * FROM rtds_prices WHERE source='chainlink' LIMIT 5", conn
)
chainlink_sample

In [ ]:
# Plot Chainlink BTC price over time
chainlink_df = pd.read_sql_query(
    "SELECT timestamp_us, price FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us",
    conn,
)
chainlink_df["time"] = pd.to_datetime(chainlink_df["timestamp_us"] / 1e6, unit="s", utc=True)

plt.figure(figsize=(12, 4))
plt.plot(chainlink_df["time"], chainlink_df["price"], linewidth=0.5, color="orange")
plt.title("Chainlink BTC/USD Oracle Price")
plt.xlabel("Time (UTC)")
plt.ylabel("Price (USD)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 4: Order Books (polymarket_books/)

Order book snapshots are stored as CSV files in `polymarket_books/`, one file per day. Each row is a snapshot of the full order book for one market at one point in time.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in microseconds |
| `interval` | Market interval (`"5m"`, `"15m"`, `"hourly"`) |
| `market_slug` | Market identifier |
| `yes_best_bid` | Best bid price for YES token |
| `yes_best_ask` | Best ask price for YES token |
| `no_best_bid` | Best bid price for NO token |
| `no_best_ask` | Best ask price for NO token |
| `yes_bids_json` | Full YES bid depth as JSON: `[[price, size], ...]` |
| `yes_asks_json` | Full YES ask depth as JSON: `[[price, size], ...]` |
| `no_bids_json` | Full NO bid depth as JSON |
| `no_asks_json` | Full NO ask depth as JSON |
| `yes_n_bids`, `yes_n_asks` | Number of bid/ask levels for YES |
| `no_n_bids`, `no_n_asks` | Number of bid/ask levels for NO |
| `yes_total_bid_size`, `yes_total_ask_size` | Total depth on each side for YES |
| `no_total_bid_size`, `no_total_ask_size` | Total depth on each side for NO |

In [ ]:
# Load order book CSV files (filenames vary by date)
books_dir = DATA_DIR / "polymarket_books"
book_files = sorted(books_dir.glob("*.csv"))
print(f"Found {len(book_files)} order book files:")
for f in book_files[:5]:
    print(f"  {f.name}")

# Load the first file as an example
if book_files:
    books_df = pd.read_csv(book_files[0])
    print(f"\nShape: {books_df.shape}")
    print(f"Columns: {list(books_df.columns)}")
    books_df.head(3)

In [ ]:
# Parse the full order book depth from JSON columns
# Each JSON column contains a list of [price, size] pairs
if book_files and not books_df.empty:
    example_row = books_df.iloc[0]
    print(f"Market: {example_row['market_slug']}")
    print(f"\nYES bids (top of book first):")
    yes_bids = json.loads(example_row["yes_bids_json"])
    for price, size in yes_bids[:5]:
        print(f"  ${price:.2f}  x  {size:.1f} shares")

    print(f"\nYES asks (cheapest first):")
    yes_asks = json.loads(example_row["yes_asks_json"])
    for price, size in yes_asks[:5]:
        print(f"  ${price:.2f}  x  {size:.1f} shares")

In [ ]:
# Plot bid-ask spread over time for one market
if book_files:
    # Load all book files and concatenate
    all_books = pd.concat([pd.read_csv(f) for f in book_files], ignore_index=True)

    # Pick a 15m market with the most snapshots
    book_counts = all_books[all_books["interval"] == "15m"].groupby("market_slug").size()
    if not book_counts.empty:
        top_slug = book_counts.idxmax()
        slug_books = all_books[all_books["market_slug"] == top_slug].copy()
        slug_books["time"] = pd.to_datetime(slug_books["timestamp_us"] / 1e6, unit="s", utc=True)
        slug_books["yes_spread"] = slug_books["yes_best_ask"] - slug_books["yes_best_bid"]

        plt.figure(figsize=(12, 4))
        plt.plot(slug_books["time"], slug_books["yes_spread"], linewidth=0.8, color="green")
        plt.title(f"YES Bid-Ask Spread - {top_slug}")
        plt.xlabel("Time (UTC)")
        plt.ylabel("Spread")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Section 5: Binance LOB (binance_lob/)

Binance BTCUSDT order book depth snapshots stored as hourly Parquet files. These provide 20-level depth at approximately 100ms resolution -- much higher frequency than the Polymarket data.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in microseconds |
| `event_time_ms` | Binance event timestamp in milliseconds |
| `ask_price_1` .. `ask_price_20` | Ask prices at levels 1-20 |
| `ask_vol_1` .. `ask_vol_20` | Ask volumes at levels 1-20 |
| `bid_price_1` .. `bid_price_20` | Bid prices at levels 1-20 |
| `bid_vol_1` .. `bid_vol_20` | Bid volumes at levels 1-20 |
| `mid_price` | (best bid + best ask) / 2 |
| `spread` | best ask - best bid |

**Note:** 20-level depth at ~100ms resolution means ~36,000 rows per hour.

In [ ]:
# Load Binance LOB Parquet files (filenames are YYYY-MM-DD_HH.parquet)
binance_dir = DATA_DIR / "binance_lob"
parquet_files = sorted(binance_dir.glob("*.parquet"))
print(f"Found {len(parquet_files)} Binance LOB files:")
for f in parquet_files[:5]:
    print(f"  {f.name}")

# Load one file to explore
if parquet_files:
    binance_sample = pd.read_parquet(parquet_files[0])
    print(f"\nShape: {binance_sample.shape}")
    print(f"Columns: {list(binance_sample.columns)}")
    binance_sample.head(3)

In [ ]:
# Plot mid_price and spread from one hour of Binance data
if parquet_files:
    binance_sample["time"] = pd.to_datetime(
        binance_sample["timestamp_us"] / 1e6, unit="s", utc=True
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(binance_sample["time"], binance_sample["mid_price"], linewidth=0.5)
    ax1.set_title("Binance BTCUSDT Mid Price (1 hour)")
    ax1.set_ylabel("Price (USD)")
    ax1.grid(True, alpha=0.3)

    ax2.plot(binance_sample["time"], binance_sample["spread"], linewidth=0.5, color="red")
    ax2.set_title("Binance BTCUSDT Spread")
    ax2.set_xlabel("Time (UTC)")
    ax2.set_ylabel("Spread (USD)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Section 6: Merging Datasets

All three data sources use `timestamp_us` (microseconds). To align them, convert to integer seconds and merge. The backtester does this automatically via `build_timeline()`, but this is useful for exploratory analysis.

**Important:** Forward-fill (`ffill`) is used because data sources update at different frequencies -- Binance updates every ~100ms, Chainlink every ~1s, and Polymarket prices every ~1s but only while a market is active.

In [ ]:
# Load all three sources and align to 1-second resolution

# 1. Market prices from SQLite
prices_df = pd.read_sql_query(
    "SELECT * FROM market_prices ORDER BY timestamp_us", conn
)
prices_df["ts_sec"] = prices_df["timestamp_us"] // 1_000_000

# 2. Chainlink prices from SQLite
chainlink_df = pd.read_sql_query(
    "SELECT * FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us", conn
)
chainlink_df["ts_sec"] = chainlink_df["timestamp_us"] // 1_000_000

# 3. Binance mid prices (aggregate to 1-second: take last value per second)
if parquet_files:
    # Load a subset of files to keep memory manageable
    binance_frames = [pd.read_parquet(f) for f in parquet_files[:3]]
    binance_df = pd.concat(binance_frames, ignore_index=True)
    binance_df["ts_sec"] = binance_df["timestamp_us"] // 1_000_000
    # Downsample to 1-second: take the last snapshot per second
    binance_1s = binance_df.groupby("ts_sec").agg(
        mid_price=("mid_price", "last"),
        spread=("spread", "last"),
    ).reset_index()
else:
    binance_1s = pd.DataFrame(columns=["ts_sec", "mid_price", "spread"])

# Merge on ts_sec
merged = prices_df.merge(
    chainlink_df[["ts_sec", "price"]].rename(columns={"price": "chainlink_btc"}),
    on="ts_sec",
    how="left",
)
merged = merged.merge(
    binance_1s[["ts_sec", "mid_price"]].rename(columns={"mid_price": "binance_btc"}),
    on="ts_sec",
    how="left",
)

# Forward-fill external prices (they update at different rates)
merged[["chainlink_btc", "binance_btc"]] = merged[["chainlink_btc", "binance_btc"]].ffill()

print(f"Merged shape: {merged.shape}")
merged[["timestamp_us", "market_slug", "yes_price", "chainlink_btc", "binance_btc"]].head(10)

## Section 7: Understanding Market Settlement

Each market resolves based on the **Chainlink oracle price**:
- Extract the market's start timestamp from its slug (e.g. `btc-updown-15m-1717243200` starts at unix time `1717243200`)
- The market's end time is `start + interval_seconds` (300 for 5m, 900 for 15m, 3600 for hourly)
- **YES wins** if Chainlink BTC price at end >= Chainlink BTC price at start
- **NO wins** if Chainlink BTC price at end < Chainlink BTC price at start

The `market_outcomes` table (if available) stores the resolved outcomes directly.

In [ ]:
# Check if market_outcomes table exists and preview it
has_outcomes = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='market_outcomes'", conn
)

if not has_outcomes.empty:
    outcomes_df = pd.read_sql_query("SELECT * FROM market_outcomes LIMIT 5", conn)
    print("market_outcomes table found:")
    display(outcomes_df)
else:
    print("market_outcomes table not found in this dataset.")
    print("You can compute outcomes from Chainlink prices instead (see below).")

In [ ]:
# Compute settlement outcome from Chainlink prices for a 15m market
import re

# Get all unique 15m slugs
slugs_15m = prices_df[prices_df["interval"] == "15m"]["market_slug"].unique()

# Load full Chainlink series for lookups
cl_full = pd.read_sql_query(
    "SELECT timestamp_us, price FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us",
    conn,
)
cl_full["ts_sec"] = cl_full["timestamp_us"] // 1_000_000

results = []
for slug in slugs_15m[:10]:  # Check first 10 markets
    m = re.match(r"btc-updown-15m-(\d+)", slug)
    if not m:
        continue
    start_ts = int(m.group(1))
    end_ts = start_ts + 900  # 15 minutes

    # Find Chainlink price nearest to start and end
    start_diff = (cl_full["ts_sec"] - start_ts).abs()
    end_diff = (cl_full["ts_sec"] - end_ts).abs()

    if start_diff.min() > 30 or end_diff.min() > 30:
        continue  # Skip if no Chainlink data within 30 seconds

    open_price = cl_full.loc[start_diff.idxmin(), "price"]
    close_price = cl_full.loc[end_diff.idxmin(), "price"]
    outcome = "YES" if close_price >= open_price else "NO"

    results.append({
        "slug": slug,
        "open_price": open_price,
        "close_price": close_price,
        "change": close_price - open_price,
        "outcome": outcome,
    })

if results:
    pd.DataFrame(results)

## Section 8: Key Notes for Strategy Development

- **The backtester merges all data automatically** -- you do not need to do the merging above yourself. Just implement `on_tick()` and the engine feeds you a `MarketState` every second.
- `state.btc_mid` = Binance BTCUSDT mid price (high-frequency reference)
- `state.chainlink_btc` = Chainlink on-chain oracle price (**used for settlement**)
- **Settlement uses Chainlink prices, not Binance.** The two can diverge by several dollars.
- Your strategy chooses which intervals to trade by filtering `state.markets` by `market.interval` (e.g. `"5m"`, `"15m"`, `"hourly"`).
- The backtester loads ALL intervals by default -- your strategy decides what to trade.
- **Order execution walks the real order book** -- if you buy 500 shares, that liquidity is consumed from the book. Large orders will get worse average prices.
- Orders submitted at tick T execute at tick T+1 (1-second latency).
- **Position limit:** 500 shares per token per market.
- **No short selling** -- you can only sell tokens you own.
- YES price + NO price should be approximately $1. Any deviation is a potential arbitrage signal.

In [ ]:
# Clean up
conn.close()

## Section 9: Backtesting a Simple Strategy - Late-Window Drift

We now reproduce the `LateWindowDrift` strategy (see `../my_strategy.py`) end-to-end in pandas so you can see every data-engineering step the backtester usually hides.

**Strategy intuition.** A binary "BTC up-or-down over the next 5 minutes" market pays $1 to whichever side Chainlink picks at close. Near expiry, the outcome is almost deterministic:

- `btc_now > btc_open`  →  YES will win
- `btc_now < btc_open`  →  NO will win

An efficient market should price the winning token near $1 in the final minute - but the Polymarket book is thin, so you often see the losing side still trading at 0.30–0.50 with a clear drift and <40% of the market's life remaining. That is the edge we try to capture.

**Rules (BTC 5m only):**

1. Capture `btc_open` from the Chainlink oracle on the first tick each market becomes active.
2. Wait until `time_remaining_frac <= 0.40` (the last 40% of the market's life).
3. Require `|btc_now − btc_open| ≥ $10` (deadband - filters out noise moves).
4. Buy the favored side if its ask is in `[0.05, 0.85]`. One entry per (market, side), 20 shares.
5. Hold until settlement; collect $1 per share if the favored side wins, $0 otherwise.

**This section focuses on data engineering.** A strategy with good intuition but dirty inputs will underperform a mediocre strategy with clean inputs.

### 9.1 Load raw data

We reopen the database (the cleanup cell above closed it) and pull only what we need:
- `market_prices` - 1 row/market/second; filter to BTC 5m slugs to keep it tiny
- `rtds_prices` (Chainlink) - our reference for both entry signal AND settlement

In [ ]:
import re
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

# Point at the data bundle you want to backtest on.
DATA_DIR = Path("../data/validation/")
DB_PATH = DATA_DIR / "polymarket.db"
assert DB_PATH.exists(), f"Run `python download_data.py` first. Missing: {DB_PATH}"

conn = sqlite3.connect(str(DB_PATH))

# Load only what we need. Filtering in SQL is MUCH faster than in pandas.
prices_raw = pd.read_sql_query(
    """
    SELECT timestamp_us, interval, market_slug,
           yes_price, no_price, yes_bid, yes_ask, no_bid, no_ask
      FROM market_prices
     WHERE interval = '5m'
       AND (market_slug LIKE 'btc-%' OR market_slug LIKE 'bitcoin-%')
     ORDER BY timestamp_us
    """,
    conn,
)

chainlink_raw = pd.read_sql_query(
    """
    SELECT timestamp_us, symbol, price
      FROM rtds_prices
     WHERE source = 'chainlink'
       AND symbol = 'BTC/USD'
     ORDER BY timestamp_us
    """,
    conn,
)
conn.close()

print(f"market_prices rows:     {len(prices_raw):,}")
print(f"chainlink rows:         {len(chainlink_raw):,}")
print(f"unique 5m BTC markets:  {prices_raw['market_slug'].nunique():,}")
print(f"time range:             "
      f"{pd.to_datetime(prices_raw['timestamp_us'].min() / 1e6, unit='s', utc=True)}  →  "
      f"{pd.to_datetime(prices_raw['timestamp_us'].max() / 1e6, unit='s', utc=True)}")
prices_raw.head(3)

### 9.2 Data cleaning - the boring part that decides everything

Four specific problems in this dataset:

1. **Two timestamp resolutions.** Both sources are in microseconds but they don't align exactly. Bucket them into integer seconds (`ts_sec = timestamp_us // 1_000_000`) before any join.
2. **Irregular Chainlink updates.** The oracle updates every ~1s but occasionally skips. A naive `merge` on `ts_sec` leaves NaNs. Fix with `reindex + ffill` onto a contiguous 1-second grid.
3. **Junk quotes.** Some rows have `yes_ask = 0.0` or `no_ask > 1.0`. Drop those so they never trigger a trade.
4. **Start time parsing.** The slug (`btc-updown-5m-1776283500`) is the only place the market's start unix timestamp lives. Parse it once and attach `start_ts` / `end_ts` / `time_remaining_frac` columns.

These five fixes are what the backtester's `build_timeline()` does under the hood - we're doing it explicitly so you can see (and modify) each step.

In [ ]:
# ── Step 1: bucket both sources into integer seconds ───────────────────────
prices = prices_raw.copy()
prices["ts_sec"] = prices["timestamp_us"] // 1_000_000

chainlink = chainlink_raw.copy()
chainlink["ts_sec"] = chainlink["timestamp_us"] // 1_000_000
# If there are multiple Chainlink updates in the same second, keep the last one.
chainlink = chainlink.groupby("ts_sec", as_index=False).agg(btc_cl=("price", "last"))

# ── Step 2: forward-fill Chainlink onto a contiguous 1-second grid ─────────
full_grid = pd.RangeIndex(
    start=int(prices["ts_sec"].min()),
    stop=int(prices["ts_sec"].max()) + 1,
    name="ts_sec",
)
chainlink_grid = (
    chainlink.set_index("ts_sec")
             .reindex(full_grid)
             .ffill()
)
# Track Chainlink staleness: seconds since the oracle actually changed.
chainlink_grid["cl_changed"] = chainlink_grid["btc_cl"].diff().abs() > 0
chainlink_grid["cl_changed"] = chainlink_grid["cl_changed"].fillna(True)
last_change_idx = np.where(chainlink_grid["cl_changed"], chainlink_grid.index, np.nan)
last_change_idx = pd.Series(last_change_idx, index=chainlink_grid.index).ffill()
chainlink_grid["cl_age_s"] = chainlink_grid.index - last_change_idx
chainlink_grid = chainlink_grid.drop(columns="cl_changed")

# ── Step 3: drop junk quotes ───────────────────────────────────────────────
before = len(prices)
prices = prices[
    (prices["yes_ask"] > 0) & (prices["yes_ask"] < 1.0) &
    (prices["no_ask"]  > 0) & (prices["no_ask"]  < 1.0) &
    (prices["yes_bid"] >= 0) & (prices["no_bid"] >= 0)
].copy()
print(f"dropped {before - len(prices):,} junk rows ({(1 - len(prices)/before) * 100:.2f}%)")

# ── Step 4: parse start_ts from slug and compute time-remaining ────────────
SLUG_RE = re.compile(r"^(?:btc|bitcoin)-updown-5m-(\d+)$")
def _parse_start(slug: str):
    m = SLUG_RE.match(slug)
    return int(m.group(1)) if m else None

# Build a slug -> (start_ts, end_ts) table once.
slug_meta = (
    prices[["market_slug"]].drop_duplicates()
           .assign(start_ts=lambda d: d["market_slug"].apply(_parse_start))
)
slug_meta = slug_meta.dropna(subset=["start_ts"])
slug_meta["start_ts"] = slug_meta["start_ts"].astype(int)
slug_meta["end_ts"]   = slug_meta["start_ts"] + 300

prices = prices.merge(slug_meta, on="market_slug", how="inner")
prices["time_remaining_s"]    = prices["end_ts"] - prices["ts_sec"]
prices["time_remaining_frac"] = prices["time_remaining_s"] / 300.0

# Drop ticks outside the lifecycle (negative time remaining = already settled).
prices = prices[
    (prices["time_remaining_frac"] > 0) &
    (prices["time_remaining_frac"] <= 1)
].copy()

# ── Step 5: join Chainlink onto the cleaned prices ─────────────────────────
merged = prices.merge(chainlink_grid, left_on="ts_sec", right_index=True, how="left")

print(f"cleaned rows:           {len(merged):,}")
print(f"unique markets:         {merged['market_slug'].nunique():,}")
print(f"null BTC prices:        {merged['btc_cl'].isna().sum():,}  "
      f"(should be small - chainlink gaps at the very start)")
merged.head(3)

### 9.3 Per-market features: `btc_open` and settlement outcome

Two derived tables we need before simulating:

- **`market_open`** - Chainlink BTC price at `start_ts` for each market. This is the reference we measure drift against.
- **`market_close`** - Chainlink BTC price at `end_ts`. `YES` wins if `close >= open`, else `NO`. This is the *ground-truth settlement* the engine uses.

We pull both directly out of the forward-filled Chainlink grid. No fuzzy matching, no nearest-neighbour search - the grid has a value for every integer second in range, which is exactly why we built it.

In [ ]:
grid_lookup = chainlink_grid["btc_cl"]  # Series indexed by ts_sec

def _grid_price(ts: int) -> float:
    """Look up Chainlink BTC at a specific second (NaN if out of range)."""
    try:
        return float(grid_lookup.loc[ts])
    except KeyError:
        return np.nan

settlements = slug_meta.copy()
settlements["btc_open"]  = settlements["start_ts"].apply(_grid_price)
settlements["btc_close"] = settlements["end_ts"].apply(_grid_price)

# Drop markets where we can't pin down open or close (edge of the dataset).
before = len(settlements)
settlements = settlements.dropna(subset=["btc_open", "btc_close"]).copy()
print(f"dropped {before - len(settlements):,} markets with missing open/close "
      f"({(1 - len(settlements)/before)*100:.1f}%)")

settlements["outcome"] = np.where(
    settlements["btc_close"] >= settlements["btc_open"], "YES", "NO"
)
print(f"markets with YES outcome:  {(settlements['outcome'] == 'YES').sum():,}  "
      f"({(settlements['outcome'] == 'YES').mean()*100:.1f}%)")
settlements.head()

### 9.4 Build the signal table (vectorised)

For every (market, tick) row we compute:

- `drift` = `btc_cl − btc_open`
- `favored` = `"YES"` if `drift > +$10`, `"NO"` if `drift < −$10`, else `None` (deadband)
- `entry_ask` = the ask of the favored side
- `in_late_window` = `time_remaining_frac <= 0.40`

All of this is a single vectorised pass - no Python loops yet.

In [ ]:
# Strategy parameters - must match my_strategy.py
LATE_WINDOW  = 0.40
DEADBAND_USD = 10.0
MAX_PAY      = 0.85
MIN_PAY      = 0.05
SIZE         = 20.0
MAX_STALE_S  = 30

# Attach btc_open and settlement outcome to every tick.
signal = merged.merge(
    settlements[["market_slug", "btc_open", "btc_close", "outcome"]],
    on="market_slug",
    how="inner",
)

signal["drift"] = signal["btc_cl"] - signal["btc_open"]

# Pick the favored side and its ask. Vectorised - no .apply.
signal["favored"] = np.where(
    signal["drift"] >  DEADBAND_USD, "YES",
    np.where(signal["drift"] < -DEADBAND_USD, "NO", None),
)
signal["entry_ask"] = np.where(
    signal["favored"] == "YES", signal["yes_ask"],
    np.where(signal["favored"] == "NO", signal["no_ask"], np.nan),
)

# The gate: all conditions the strategy requires to fire.
signal["eligible"] = (
    (signal["favored"].notna()) &
    (signal["time_remaining_frac"] <= LATE_WINDOW) &
    (signal["entry_ask"] > MIN_PAY) &
    (signal["entry_ask"] < MAX_PAY) &
    (signal["cl_age_s"] <= MAX_STALE_S)
)

eligible = signal[signal["eligible"]]
print(f"eligible signal ticks: {len(eligible):,}  "
      f"across {eligible['market_slug'].nunique():,} markets")
eligible[[
    "ts_sec", "market_slug", "time_remaining_frac",
    "btc_cl", "btc_open", "drift", "favored", "entry_ask", "outcome",
]].head()

### 9.5 Simulate entries with T+1 latency and cash/position limits

The backtester engine enforces three realism rules:

1. **T+1 latency** - an order submitted at tick `T` fills at tick `T+1` using T+1's ask.
2. **Cash limit** - can't enter a position if we don't have the cash.
3. **One entry per (market, side)** - no stacking into the same idea.

We do the same, per market. Pandas `groupby + apply` is fine here: even with 1000+ markets, this is fast.

In [ ]:
# For each market, find the FIRST eligible tick per favored side.
# That's where we'd place the order. Fill price uses the NEXT tick's ask (T+1 latency).
entries_per_market = []

# Index signal by (slug, ts_sec) for fast "next tick ask" lookup.
signal_sorted = signal.sort_values(["market_slug", "ts_sec"]).reset_index(drop=True)
# Shift yes_ask / no_ask UP by one within each market to get "T+1 ask at tick T".
signal_sorted["yes_ask_t1"] = signal_sorted.groupby("market_slug")["yes_ask"].shift(-1)
signal_sorted["no_ask_t1"]  = signal_sorted.groupby("market_slug")["no_ask"].shift(-1)

fill_price = np.where(
    signal_sorted["favored"] == "YES", signal_sorted["yes_ask_t1"],
    np.where(signal_sorted["favored"] == "NO", signal_sorted["no_ask_t1"], np.nan),
)
signal_sorted["fill_price"] = fill_price

# Eligibility AFTER knowing the T+1 fill price is valid.
signal_sorted["eligible_fillable"] = (
    signal_sorted["eligible"] &
    signal_sorted["fill_price"].notna() &
    (signal_sorted["fill_price"] > MIN_PAY) &
    (signal_sorted["fill_price"] < MAX_PAY)
)

# First eligible tick per (market, favored side).
first_entries = (
    signal_sorted[signal_sorted["eligible_fillable"]]
        .groupby(["market_slug", "favored"], as_index=False)
        .first()
        [["ts_sec", "market_slug", "favored", "fill_price", "outcome"]]
)
first_entries = first_entries.rename(columns={"fill_price": "entry_price"})
first_entries["cost"]   = first_entries["entry_price"] * SIZE
first_entries["payoff"] = np.where(
    first_entries["favored"] == first_entries["outcome"], 1.0 * SIZE, 0.0
)
first_entries["pnl"]    = first_entries["payoff"] - first_entries["cost"]

print(f"candidate entries: {len(first_entries):,}")
first_entries.head()

### 9.6 Walk the trade tape chronologically with a $10,000 cash budget

Up to this point everything has been vectorised. But cash is a stateful constraint - we need to walk entries in time order and skip ones we can't afford. This is the *only* loop in the whole backtest.

In [ ]:
STARTING_CASH = 10_000.0

entries_sorted = first_entries.sort_values("ts_sec").reset_index(drop=True)

cash = STARTING_CASH
open_positions = []  # list of (settle_ts, payoff)
taken = []

for row in entries_sorted.itertuples(index=False):
    # Settle any positions whose end_ts has passed this entry's ts_sec.
    still_open = []
    for settle_ts, payoff in open_positions:
        if settle_ts <= row.ts_sec:
            cash += payoff
        else:
            still_open.append((settle_ts, payoff))
    open_positions = still_open

    if cash < row.cost:
        continue  # can't afford - skip
    cash -= row.cost

    end_ts = int(settlements.loc[settlements["market_slug"] == row.market_slug, "end_ts"].iloc[0])
    open_positions.append((end_ts, row.payoff))
    taken.append({
        "ts_sec": row.ts_sec,
        "market_slug": row.market_slug,
        "favored": row.favored,
        "entry_price": row.entry_price,
        "cost": row.cost,
        "outcome": row.outcome,
        "payoff": row.payoff,
        "pnl": row.pnl,
        "cash_after_entry": cash,
        "end_ts": end_ts,
    })

# Settle everything that's still open.
for _settle_ts, payoff in open_positions:
    cash += payoff

trades = pd.DataFrame(taken)
final_pv  = cash
total_pnl = final_pv - STARTING_CASH

print(f"trades taken:             {len(trades):,}")
print(f"starting cash:            ${STARTING_CASH:,.2f}")
print(f"final portfolio value:    ${final_pv:,.2f}")
print(f"total P&L:                ${total_pnl:+,.2f}")
if len(trades):
    win_rate = (trades["pnl"] > 0).mean() * 100
    print(f"win rate:                 {win_rate:.1f}%   "
          f"(avg win ${trades.loc[trades['pnl']>0,'pnl'].mean():.2f}, "
          f"avg loss ${trades.loc[trades['pnl']<0,'pnl'].mean():.2f})")
trades.head()

### 9.7 Equity curve and sanity checks

In [ ]:
if len(trades):
    # Build an equity curve: cash at each settlement event (entry debit + payout credit).
    events = []
    for t in trades.itertuples(index=False):
        events.append((t.ts_sec, -t.cost))          # buy: cash out
        events.append((t.end_ts, t.payoff))         # settle: cash in
    events_df = pd.DataFrame(events, columns=["ts_sec", "delta"]).sort_values("ts_sec")
    events_df["cum_cash"] = STARTING_CASH + events_df["delta"].cumsum()
    events_df["time"] = pd.to_datetime(events_df["ts_sec"], unit="s", utc=True)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    ax1.plot(events_df["time"], events_df["cum_cash"], linewidth=1.2)
    ax1.axhline(STARTING_CASH, linestyle="--", color="gray", linewidth=0.7)
    ax1.set_title(f"Equity curve - Late-Window Drift  (final P&L ${total_pnl:+,.2f})")
    ax1.set_ylabel("Portfolio value (USD)")
    ax1.grid(True, alpha=0.3)

    ax2.hist(trades["pnl"], bins=40, edgecolor="black")
    ax2.axvline(0, color="red", linewidth=1)
    ax2.set_title(f"Per-trade P&L distribution  (n={len(trades):,})")
    ax2.set_xlabel("P&L per trade (USD)")
    ax2.set_ylabel("Count")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No trades were taken - loosen the thresholds and rerun Section 9.4+.")

### 9.8 Why numbers may differ slightly from `run_backtest.py`

Two small, expected sources of drift between this notebook and the CLI backtester:

- **Order book.** The backtester walks the actual JSON order book in `polymarket_books/*.csv`, so large orders eat depth and get worse average prices. We use the top-of-book ask from `market_prices`, which is the best available price only.
- **Snapshot cadence.** The engine ticks once per second; we use only ticks where Polymarket emitted a row, which is ~1 Hz but not perfectly gap-free.

Both effects are small for a 20-share position. If you crank `SIZE` up to 200+, the CLI backtester will show worse fills; the notebook won't. **Trust the CLI number for submission scoring.** The notebook is for understanding the signal.